In [ ]:
import pandas as pd
df = pd.read_csv('abalone.csv')
abalone = df

abalone.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def preprocess_data(abalone):
    sex_encoder = LabelEncoder()
    abalone['Sex'] = sex_encoder.fit_transform(abalone['Sex'])  # Encoding categorical variable 'Sex'
    # 1 is infant, 2 is male, 0 is female

    X = abalone.drop(columns=['Diameter'])  
    y = abalone['Diameter'] 

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    numerical_features = ['Length', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']

    numerical_transformer = StandardScaler()
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features)
        ], remainder='passthrough')
    
    
    X_train_preprocessed = preprocessor.fit_transform(X_train)
    X_test_preprocessed = preprocessor.transform(X_test)

    return X_train_preprocessed, X_test_preprocessed, y_train, y_test


X_train_preprocessed, X_test_preprocessed, y_train, y_test = preprocess_data(abalone)

abalone


In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error

def tune_xgboost(X_train, y_train):  
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [1,2],
        'learning_rate': [0.01, 0.1],
        'gamma': [0, 0.1],
        'subsample': [0.6, 0.8],
        'colsample_bytree': [0.6, 0.8]
    }
    xgb_regressor = XGBRegressor(random_state=42) 
    grid_search = GridSearchCV(estimator=xgb_regressor, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train) 
    print("Best Parameters:", grid_search.best_params_) 
    return grid_search.best_estimator_ 

X_train_preprocessed, X_test_preprocessed, y_train, y_test = preprocess_data(abalone) 
best_xgboost_model = tune_xgboost(X_train_preprocessed, y_train) 
xgboost_predictions = best_xgboost_model.predict(X_test_preprocessed) 
mse_xgboost = mean_squared_error(y_test, xgboost_predictions) 
print("Mean Squared Error for XGBoost:", mse_xgboost)

# Quiz Question
everything above is me re-inserting my old data model from the last assignment 

In [ ]:

from sklearn.decomposition import PCA
import numpy as np

pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_preprocessed)
X_test_pca = pca.transform(X_test_preprocessed)


best_xgboost_model_pca = tune_xgboost(X_train_pca, y_train)


xgboost_predictions_pca = best_xgboost_model_pca.predict(X_test_pca)


test_rmse_pca = np.sqrt(mean_squared_error(y_test, xgboost_predictions_pca))
print("RMSE for testing data with PCA:", test_rmse_pca)
